## **Task 1 : Implement Quantization from Scratch**

In [2]:
import numpy as np

# Function to quantize float32 values into int8
def quantize_to_int8(data):

    # Convert input list into numpy float32 array
    data = np.array(data, dtype=np.float32)

    # Find minimum and maximum values
    rmin = data.min()
    rmax = data.max()

    # INT8 range
    qmin = -128
    qmax = 127

    # Calculate scale factor
    scale = (rmax - rmin) / (qmax - qmin)

    # Calculate zero-point
    zero_point = round(qmin - rmin / scale)

    # Quantize values
    quantized = np.round(data / scale + zero_point)

    # Ensure values remain within INT8 range
    quantized = np.clip(quantized, qmin, qmax)

    return quantized.astype(np.int8), scale, zero_point


# Sample input data
arr = [1.2, 2.5, 3.7, 5.1, 6.8]

# Perform quantization
q_arr, scale, zp = quantize_to_int8(arr)

# Display results
print("Original Values :", arr)
print("Quantized Values:", q_arr)
print("Scale           :", scale)
print("Zero Point      :", zp)

Original Values : [1.2, 2.5, 3.7, 5.1, 6.8]
Quantized Values: [-128  -69  -15   49  127]
Scale           : 0.021960786
Zero Point      : -183


### Observation

- Floating point values were successfully converted into INT8 values.
- The values occupy less memory compared to FP32 representation.
- Small precision loss occurs because of rounding during quantization.

### **Task 2: Estimate Model Size from Parameter Count**

In [5]:
# Function to estimate model size
def model_size(params, dtype):

    # Number of bytes used by each datatype
    bytes_map = {
        "fp32": 4,
        "fp16": 2,
        "int8": 1
    }

    # Calculate total bytes occupied by parameters
    total_bytes = params * bytes_map[dtype]

    # Convert into MB and GB
    size_mb = total_bytes / (1024 ** 2)
    size_gb = total_bytes / (1024 ** 3)

    return size_mb, size_gb


# Take input from user
params = int(input("Enter number of parameters: "))
dtype = input("Enter datatype (fp32/fp16/int8): ").lower()

# Calculate model size
mb, gb = model_size(params, dtype)

# Display result
print("\nModel Size Estimation")
print("----------------------")
print("Parameters :", f"{params:,}")
print("Datatype   :", dtype)
print("Size (MB)  :", round(mb, 2))
print("Size (GB)  :", round(gb, 2))

Enter number of parameters: 7_000_000_000
Enter datatype (fp32/fp16/int8): fp16

Model Size Estimation
----------------------
Parameters : 7,000,000,000
Datatype   : fp16
Size (MB)  : 13351.44
Size (GB)  : 13.04


### Observation

- Model size increases linearly with the number of parameters.
- FP32 requires 4 bytes per parameter while INT8 requires only 1 byte.
- Therefore, INT8 quantization reduces storage requirements by approximately 75%.
- Smaller model size allows deployment on GPUs with lower memory capacity.

## **Task 3: System Design Scenario – Multi-GPU Inference**

For a large language model deployment, I would first distribute the model across GPUs within the same node before distributing it across multiple nodes.

A node contains 8 GPUs connected using NVLink. NVLink provides extremely high bandwidth and low latency communication between GPUs. Because of this, GPUs can exchange activations, attention outputs, and intermediate tensors very efficiently.

The first level of model parallelism should therefore occur inside a node. Consecutive transformer layers can be split among GPUs connected through NVLink. Since communication is frequent during inference, keeping these transfers inside NVLink minimizes delays.

Only when the model becomes larger than the total memory available inside one node would I extend the model across multiple nodes. Communication between nodes occurs through InfiniBand. Although InfiniBand is fast for cluster communication, it is slower and has higher latency compared to NVLink.

###**Role of NVLink**
* High bandwidth GPU-to-GPU communication
* Low latency
* Ideal for tensor parallelism
* Ideal for pipeline parallelism inside a node

###**Role of InfiniBand**
* Communication between different nodes
* Distributed inference coordination
*  transfer across servers
* Large-scale cluster communication

###**What Happens if Roles Are Swapped?**

If frequent tensor communication is routed through InfiniBand instead of NVLink:

Communication latency increases significantly.
GPUs spend more time waiting for data.
Throughput decreases.
Inference response time increases.

If NVLink is expected to connect separate nodes:

It cannot provide node-to-node connectivity.
The architecture becomes impractical because NVLink is designed primarily for intra-node communication.

## **Task 4: Compute-Bound vs Memory-Bound Estimation**

In [7]:

def bound_type(flops_per_token, bytes_per_token, compute_flops_per_sec, mem_bandwidth_bytes_per_sec):
    """
    Determine whether inference for one token is compute-bound or memory-bound.

    flops_per_token             : FLOPs required to produce one output token
    bytes_per_token              : bytes that must be read from memory per token
                                    (e.g. model weights read during autoregressive decode)
    compute_flops_per_sec        : GPU peak compute throughput (FLOPs/sec)
    mem_bandwidth_bytes_per_sec  : GPU memory bandwidth (bytes/sec)

    Returns a dict with the verdict and the two time estimates.
    """
    time_compute = flops_per_token / compute_flops_per_sec
    time_memory = bytes_per_token / mem_bandwidth_bytes_per_sec

    if time_compute > time_memory:
        verdict = "compute-bound"
    elif time_memory > time_compute:
        verdict = "memory-bound"
    else:
        verdict = "balanced (roofline ridge point)"

    # Arithmetic intensity = FLOPs per byte moved; useful diagnostic
    arithmetic_intensity = flops_per_token / bytes_per_token
    ridge_point = compute_flops_per_sec / mem_bandwidth_bytes_per_sec

    return {
        "verdict": verdict,
        "time_compute_s": time_compute,
        "time_memory_s": time_memory,
        "arithmetic_intensity_flops_per_byte": arithmetic_intensity,
        "ridge_point_flops_per_byte": ridge_point,
    }


In [8]:
# Example configs
# GPU spec roughly modeled on an A100-class card: ~312 TFLOPs (fp16 tensor), ~2 TB/s HBM bandwidth
A100_COMPUTE = 312e12       # FLOPs/sec
A100_MEM_BW  = 2.0e12       # bytes/sec

configs = {
    # Small model, single-token autoregressive decode (batch size 1):
    # ~2 FLOPs per parameter per token (standard transformer estimate), bytes read ~= model size in fp16
    "1B model, batch=1 decode": dict(
        flops_per_token=2 * 1e9,
        bytes_per_token=1e9 * 2,   # fp16 -> 2 bytes/param, whole model read per token
        compute_flops_per_sec=A100_COMPUTE,
        mem_bandwidth_bytes_per_sec=A100_MEM_BW,
    ),
    # Large model, single-token decode
    "70B model, batch=1 decode": dict(
        flops_per_token=2 * 70e9,
        bytes_per_token=70e9 * 2,
        compute_flops_per_sec=A100_COMPUTE,
        mem_bandwidth_bytes_per_sec=A100_MEM_BW,
    ),
    # Same 70B model but large batch (batched decode): weight read amortized across many tokens in the batch
    "70B model, batch=256 decode": dict(
        flops_per_token=2 * 70e9,
        bytes_per_token=(70e9 * 2) / 256,
        compute_flops_per_sec=A100_COMPUTE,
        mem_bandwidth_bytes_per_sec=A100_MEM_BW,
    ),
}

for name, cfg in configs.items():
    result = bound_type(**cfg)
    print(f"--- {name} ---")
    for k, v in result.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.6g}")
        else:
            print(f"  {k}: {v}")
    print()


--- 1B model, batch=1 decode ---
  verdict: memory-bound
  time_compute_s: 6.41026e-06
  time_memory_s: 0.001
  arithmetic_intensity_flops_per_byte: 1
  ridge_point_flops_per_byte: 156

--- 70B model, batch=1 decode ---
  verdict: memory-bound
  time_compute_s: 0.000448718
  time_memory_s: 0.07
  arithmetic_intensity_flops_per_byte: 1
  ridge_point_flops_per_byte: 156

--- 70B model, batch=256 decode ---
  verdict: compute-bound
  time_compute_s: 0.000448718
  time_memory_s: 0.000273438
  arithmetic_intensity_flops_per_byte: 256
  ridge_point_flops_per_byte: 156



# **Observation**
* Compute-bound workloads benefit from faster GPUs, optimized kernels, tensor cores, and lower computational complexity.
* Memory-bound workloads benefit from quantization, better caching, reduced memory transfers, and higher memory bandwidth.
* Identifying the bottleneck helps engineers focus optimization efforts on the actual limiting factor rather than applying unnecessary optimizations